<a href="https://colab.research.google.com/github/borysovamaryna/a-b-test-sql-analytics-/blob/main/ab_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scipy pandas numpy

import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/bq-results-20260407-175108-1775584499051
ab_test_df = pd.read_csv("ab_test.csv")
ab_test_df.head()

Mounted at /content/drive
/content/drive/MyDrive/bq-results-20260407-175108-1775584499051


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-05,Cambodia,desktop,Asia,Undefined,2,1,new account,1
1,2020-11-05,Cambodia,desktop,Asia,Undefined,1,1,new account,1
2,2021-01-19,Cambodia,tablet,Asia,Organic Search,4,2,new account,1
3,2021-01-09,Kazakhstan,mobile,Asia,Paid Search,4,1,new account,1
4,2020-12-15,Qatar,desktop,Asia,Direct,4,1,new account,1


In [ ]:
print(ab_test_df['event_name'].unique())

['new account' 'session with orders' 'session' 'user_engagement'
 'session_start' 'page_view' 'first_visit' 'scroll' 'view_item'
 'view_promotion' 'add_shipping_info' 'add_to_cart' 'begin_checkout'
 'add_payment_info' 'view_search_results' 'select_promotion' 'select_item'
 'click' 'view_item_list']


In [ ]:
pivot_df = ab_test_df.pivot_table(
    index=['test', 'test_group'],
    columns='event_name',
    values='value',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Задаємо назви метрикам

metrics = {
    'add_payment_info_rate': 'add_payment_info',
    'add_shipping_info_rate': 'add_shipping_info',
    'begin_checkout_rate': 'begin_checkout',
    'new_accounts_rate': 'new account'
}

# проводимо розрахунки z-test

def z_test(success_a, size_a, success_b, size_b):
    if size_a == 0 or size_b == 0:
        return 0, 1

    p1 = success_a / size_a
    p2 = success_b / size_b

    p_pool = (success_a + success_b) / (size_a + size_b)

    se = np.sqrt(p_pool * (1 - p_pool) * (1/size_a + 1/size_b))

    if se == 0:
        return 0, 1

    z = (p1 - p2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    return z, p_value

# Основний розрахунок


results = []

for test_name in pivot_df['test'].unique():

    test_df = pivot_df[pivot_df['test'] == test_name]

    groups = sorted(test_df['test_group'].unique())

    if len(groups) != 2:
        continue

    g1, g2 = groups

    group_a = test_df[test_df['test_group'] == g1].sum(numeric_only=True)
    group_b = test_df[test_df['test_group'] == g2].sum(numeric_only=True)

    for metric_name, metric in metrics.items():

        success_a = group_a[metric]
        size_a = group_a['session']

        success_b = group_b[metric]
        size_b = group_b['session']

        # конверсії
        conv_a = success_a / size_a if size_a > 0 else 0
        conv_b = success_b / size_b if size_b > 0 else 0

        # різниця %
        uplift = ((conv_b / conv_a) - 1) * 100 if conv_a > 0 else 0

        # z-test
        z, p = z_test(success_a, size_a, success_b, size_b)

        results.append({
            'test': test_name,

            # метрика
            'metric': metric_name,

            # ділене та дільник
            'numerator': metric,
            'denominator': 'session',

            # group A
            'group_A_numerator': success_a,
            'group_A_denominator': size_a,
            'group_A_conversion': conv_a,

            # group B
            'group_B_numerator': success_b,
            'group_B_denominator': size_b,
            'group_B_conversion': conv_b,

            # показники результату
            'uplift_%': uplift,
            'z_score': z,
            'p_value': p,
            'significant_95': p < 0.05
        })

# Фінальна таблиця

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(['test', 'metric']).reset_index(drop=True)

print(result_df)


    test                  metric          numerator denominator  \
0      1   add_payment_info_rate   add_payment_info     session   
1      1  add_shipping_info_rate  add_shipping_info     session   
2      1     begin_checkout_rate     begin_checkout     session   
3      1       new_accounts_rate        new account     session   
4      2   add_payment_info_rate   add_payment_info     session   
5      2  add_shipping_info_rate  add_shipping_info     session   
6      2     begin_checkout_rate     begin_checkout     session   
7      2       new_accounts_rate        new account     session   
8      3   add_payment_info_rate   add_payment_info     session   
9      3  add_shipping_info_rate  add_shipping_info     session   
10     3     begin_checkout_rate     begin_checkout     session   
11     3       new_accounts_rate        new account     session   
12     4   add_payment_info_rate   add_payment_info     session   
13     4  add_shipping_info_rate  add_shipping_info     sessio

In [ ]:
result_df.to_csv(
    '/content/drive/MyDrive/ab_test_results.csv',
    index=False,
    sep=';'
)

#Посилання на файл з даними

https://drive.google.com/file/d/1Gccoj-xTjOr9KZLwq-Vy5Odk-ZLtxOhr/view?usp=sharing


#Посилання на дашборд

https://public.tableau.com/app/profile/maryna.borysova/viz/ABtest_17756611762510/ABtest?publish=yes